<a href="https://colab.research.google.com/github/umang0015/Ai-DataScience/blob/main/14AugSession_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# import libraries  for measuring scores , tensorflow
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os , random
os.environ['TF_CPP_MIN_LOG_LEVEL']= '3'
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
import keras
import keras_hub

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report ,confusion_matrix
SEED=42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
keras.utils.set_random_seed(SEED)

# check keras and hub versions
print(keras.__version__)
print(keras_hub.__version__)

3.13.2
0.26.0


In [3]:
df = pd.read_csv('/content/drive/MyDrive/Models/SPAM text - SPAM text.csv')
df

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."
...,...,...
5567,spam,This is the 2nd time we have tried 2 contact u...
5568,ham,Will ü b going to esplanade fr home?
5569,ham,"Pity, * was in mood for that. So...any other s..."
5570,ham,The guy did some bitching but I acted like i'd...


In [4]:
# print shape , remove duplicates if any , check for missing values
# when you drop duplicates rows , index numbers might not change , better to reset
print(df.shape)
df = df.drop_duplicates().reset_index(drop=True)
print(df.shape)
print(df.isnull().sum())

(5572, 2)
(5157, 2)
Category    0
Message     0
dtype: int64


In [5]:
# label encode the category column
df['label'] = (df['Category'] == 'spam').astype(int)

In [6]:
# split into input and output columns
X = df['Message'].values
Y = df['label'].values

# train test split
X_train, X_test , y_train ,y_test = train_test_split(X,Y,test_size=0.2, random_state=42)
#

In [7]:
# select bertclassifier from kera_hub,
# create object by setting parameterSampler
spam_classifier = keras_hub.models.BertTextClassifier.from_preset(
    "bert_tiny_en_uncased", num_classes=2)
spam_classifier.preprocessor.seq_length = 40
spam_classifier.summary()

# bert_base_en_uncased
# bert_small_en_uncased

Preprocessor: "bert_text_classifier_preprocessor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                                                  ┃                                   Config ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ bert_tokenizer (BertTokenizer)                                │                       Vocab size: 30,522 │
└───────────────────────────────────────────────────────────────┴──────────────────────────────────────────┘

Model: "bert_text_classifier"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ padding_mask (InputLayer)     │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ segment_ids (InputLayer)      │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ token_ids (InputLayer)        │ (None, None)              │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ bert_backbone (BertBackbone)  │ [(None, 128), (None,      │       4,385,920 │ padding_mask[0][0],        │
│                               │ None, 128)]               │                 │ segment_ids[0][0],         │
│                               │                           │                 │ token_ids[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ classifier_dropout (Dropout)  │ (None, 128)               │               0 │ bert_backbone[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ logits (Dense)                │ (None, 2)                 │             258 │ classifier_dropout[0][0]   │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 4,386,178 (16.73 MB)

 Trainable params: 4,386,178 (16.73 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# BERT uses "WORDPIECE" toekenization , instead of one id per word
# rare word get split into smaller known pieces


In [9]:
# apply this tokenizaton
tokenizer = spam_classifier.preprocessor.tokenizer

In [10]:
# trey tokenizer on sample data
samples = ["call now for your free ringtone " "nah i dont think he is going to usf"]
for s in samples:
  id = tokenizer(s)
  pieces =[tokenizer.id_to_token(i) for i in id]
  print(s)
  # print(id)
  print(pieces)

call now for your free ringtone nah i dont think he is going to usf
['call', 'now', 'for', 'your', 'free', 'ring', '##tone', 'nah', 'i', 'don', '##t', 'think', 'he', 'is', 'going', 'to', 'us', '##f']


In [11]:
# give me the weightage for the classes
# as spam messages has lower class count compared to ham
# apply equal weights formulation to treat both class same, avoid bias
# class weight in BERT is very important , without that it can not focus on imbalanced data
n_ham = (y_train== 0).sum()
n_spam = (y_train == 1).sum()
print(n_ham , n_spam)

# calculate the weight for both classes

class_weights={0:len(y_train ) / (2 * n_ham),1:len(y_train) / (2*n_spam)}
print(class_weights)

# spam get higher weights so that model focus more on spam


3620 505
{0: np.float64(0.5697513812154696), 1: np.float64(4.084158415841584)}


In [12]:
# define compiler and build the model
spam_classifier.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0005),
    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

In [13]:
# train the model
spam_classifier.fit(
    x=X_train,
    y=y_train,
    epochs=2,
    class_weight = class_weights ,
    batch_size=32
)

Epoch 1/2
129/129 ━━━━━━━━━━━━━━━━━━━━ 41s 150ms/step - accuracy: 0.9396 - loss: 0.2058
Epoch 2/2
129/129 ━━━━━━━━━━━━━━━━━━━━ 6s 45ms/step - accuracy: 0.9804 - loss: 0.0741


In [17]:
# sample test on new sample message
logits= spam_classifier.predict(["congratulations ! claim your free prize of 1000. reply to 85888778"])
prob = np.array(keras.ops.softmax(logits,axis=-1))[:,1]
if prob > 0.5:
  print("Spam")
else:
  print("ham")


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 683ms/step
Spam
